# RSNA Knee — train DINOv2 ViT-S

Dual T4, 12 h session. Attach: competition data, the **labels** notebook output, `metaresearch/dinov2` (PyTorch, small). Internet off.

Runs resume: set `PREV_DIRS` to this notebook's previous output to add folds in a new session.


In [ ]:
TAG, TIMM_NAME = "dino", "vit_small_patch14_dinov2.lvd142m"
MODEL_DIR = "/kaggle/input/models/metaresearch/dinov2/pytorch/small/1"
UNFREEZE, BATCH_STUDIES, ACCUM, LR_BB = 8, 6, 3, 5e-5     # the configuration that scored LB 0.842
FOLD_SECONDS = 95 * 60        # measured 94-95 min per fold (DINO); CNX re-measures itself
SEEDS = [42, 7, 11, 23]                                        # ~6 folds fit: all of seed 42, then seed 7

DATA_DIR = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
WORK     = "/kaggle/working"
TARGETS_DIR = "/kaggle/input/notebooks/kiritohayate95/rsna-knee-abnormality-detection-weak-labeling"   # output of labels.ipynb
# Continue across sessions: attach this notebook's own previous output and list it here.
# Finished (seed, fold) checkpoints are copied over and skipped; training resumes at the next one.
PREV_DIRS = []   # e.g. ["/kaggle/input/notebooks/kiritohayate95/<train-dino>"]

IMG_SIZE, CROP_MM = 336, 130.0          # identical in every notebook: C feeds one pixel tensor to all
SLOT_DEFS  = [("axial", 1), ("sagittal", 0), ("coronal", 1), ("sagittal", 1)]
K_PER_SLOT = [8, 12, 10, 16]
FP_FIELDS  = ["Manufacturer", "ManufacturerModelName", "SoftwareVersions", "ReceiveCoilName"]

N_FOLDS, FOLDS = 5, [0, 1, 2, 3, 4]
HEAD_EPOCHS, PHASE_B_EPOCHS = 120, 5
LR_HEAD, WD, EMA_DECAY, GOLD_W = 1e-3, 0.01, 0.99, 3.0
HEAD_HIDDEN, HEAD_DROP = 256, 0.2
PRIOR_W, MAX_MIX, LAB_MIX, TXT_W = 1.0, 0.30, 0.15, 0.10
EXTRACT_BATCH, NUM_WORKERS, HDR_THREADS, JPEG_Q = 8, 4, 8, 92
TIME_BUDGET  = 11.8 * 3600    # output is only ~300 MB of checkpoints, saved in a few minutes

### Setup


In [ ]:
import os, re, json, glob, time, shutil, warnings
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, cv2, pydicom
import torch, torch.nn as nn, torch.nn.functional as F, timm
from torch.utils.data import Dataset, DataLoader
from timm.utils import ModelEmaV3
from sklearn.metrics import roc_auc_score
from timm.layers import resample_abs_pos_embed
warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True
cv2.setNumThreads(0)                     # DataLoader workers own the 4 vCPUs

DEV, GPU_IDS = "cuda", [0, 1]
TRAIN_IMG = f"{DATA_DIR}/train_series"
N_TOK     = sum(K_PER_SLOT)
TOK2SLOT  = np.concatenate([np.full(k, i, dtype=np.int64) for i, k in enumerate(K_PER_SLOT)])
T0 = time.time()

sub_df   = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
tr_ser   = pd.read_csv(f"{DATA_DIR}/train_series.csv")
ID, SER, REPORT = "StudyInstanceUID", "SeriesInstanceUID", "Report"
LABELS = [c for c in sub_df.columns if c != ID]
NL = len(LABELS)
train_df["is_gold"] = train_df[LABELS].notna().all(axis=1).astype(int)
SERPATH, DCM_GLOB = "{img}/{sid}/{ser}", "*.dcm"


### Series metadata
One header per series, 16 threads: plane from the slice normal, fluid-sensitivity
from TE/TR/TI, laterality, and the scanner fingerprint that groups the folds.


In [ ]:
ser_all = tr_ser.assign(split="train")

def read_hdr(args):
    sid, ser, sp = args
    d0 = SERPATH.format(img=TRAIN_IMG, sid=sid, ser=ser)
    files = sorted(glob.glob(os.path.join(d0, DCM_GLOB)))
    if not files:
        return None
    try:
        d = pydicom.dcmread(files[len(files) // 2], stop_before_pixels=True, force=True)
    except Exception:
        return None
    iop, plane = getattr(d, "ImageOrientationPatient", None), "unknown"
    if iop is not None and len(iop) == 6:
        v = np.abs(np.cross([float(x) for x in iop[:3]], [float(x) for x in iop[3:]]))
        plane = ["sagittal", "coronal", "axial"][int(np.argmax(v))]
    return dict(
        split=sp, sid=sid, ser=ser, n=len(files), plane=plane, dir=d0,
        desc=str(getattr(d, "SeriesDescription", "")),
        lat=str(getattr(d, "Laterality", "") or getattr(d, "ImageLaterality", "")),
        fp="|".join(str(getattr(d, a, "")) for a in FP_FIELDS),
        TR=float(getattr(d, "RepetitionTime", 0) or 0),
        TE=float(getattr(d, "EchoTime", 0) or 0),
        TI=float(getattr(d, "InversionTime", 0) or 0))

with ThreadPoolExecutor(max_workers=16) as ex:
    meta = pd.DataFrame([r for r in ex.map(read_hdr, zip(ser_all[ID], ser_all[SER], ser_all.split))
                         if r is not None])

FS_KW = r"(stir|fs\b|fatsat|fat[_\- ]?sat|spair|spir|_fs|t2|tirm)"
meta["fluid"] = meta.desc.str.lower().str.contains(FS_KW, regex=True, na=False).astype(int)
meta.loc[(meta.TE > 40) & (meta.TR > 1500), "fluid"] = 1
meta.loc[(meta.TE < 20) & (meta.TR < 900),  "fluid"] = 0
meta.loc[meta.TI > 0, "fluid"] = 1
for col in ser_all.columns:
    if "fluid" in col.lower():
        meta["fluid"] = meta.ser.map(ser_all.set_index(SER)[col].to_dict()).fillna(meta.fluid).astype(int)
    if "plane" in col.lower():
        meta["plane"] = meta.ser.map(ser_all.set_index(SER)[col].astype(str).str.lower().to_dict()).fillna(meta.plane)

meta["slot"] = -1
for i, (pl, fl) in enumerate(SLOT_DEFS):
    meta.loc[(meta.plane == pl) & (meta.fluid == fl), "slot"] = i

meta["L"] = meta.lat.str.upper().str[:1]
meta.loc[~meta.L.isin(["L", "R"]), "L"] = ""
dl = meta.desc.str.lower()
meta.loc[(meta.L == "") & dl.str.contains(r"\b(left|lt|links|izquierd|sol|gauche|linker)\b",
         regex=True, na=False), "L"] = "L"
meta.loc[(meta.L == "") & dl.str.contains(r"\b(right|rt|rechts|derech|sag|droit|rechter)\b",
         regex=True, na=False), "L"] = "R"

smeta = []
for sid, g in meta.groupby("sid"):
    f = g.fp.mode().iloc[0]
    smeta.append({ID: sid, "fp": f if re.search(r"[A-Za-z0-9]", f) else "anon:" + sid})
train_df = train_df.merge(pd.DataFrame(smeta), on=ID, how="left")
train_df["fp"] = train_df.fp.fillna("unk")
print(f"{len(meta)} series | {train_df.fp.nunique()} scanner groups | {(time.time()-T0)/60:.1f} min")


### Training targets
Read from the labels notebook: soft targets, gold flags and the report n-gram features
that Phase A distils. Gold studies are weighted by `GOLD_W`.


In [ ]:
# Targets come from the labels notebook, so both architectures train on identical labels.
T = pd.read_csv(f"{TARGETS_DIR}/targets.csv").set_index(ID).loc[train_df[ID]]
soft  = T[[f"soft::{L}" for L in LABELS]].values.astype(np.float32)
txt_t = T[[c for c in T.columns if c.startswith("txt_")]].values.astype(np.float32)
sw    = np.where(T.is_gold.values[:, None] == 1, GOLD_W, 1.0).astype(np.float32).repeat(NL, 1)
pc    = np.clip(soft, 1e-6, 1 - 1e-6)
FLOOR = float((-(pc * np.log(pc) + (1 - pc) * np.log(1 - pc)) * sw).mean())
info  = json.load(open(f"{TARGETS_DIR}/targets_info.json"))
print(f"targets: {info['source']} (gold {info['gold_auc'][info['source']]:.4f}) | loss floor {FLOOR:.3f}")


### Backbone
`PoolWrap` is byte-identical to notebook C.


In [ ]:
bb = timm.create_model(TIMM_NAME, pretrained=False, num_classes=0, global_pool="", img_size=IMG_SIZE)
tgt = bb.state_dict()

# Kaggle ships DINOv2 in HF transformers layout: q/k/v are separate matrices that timm fuses into
# one qkv. Loaded unfused they stay random, and strict=False reports that only as a warning.
raw = torch.load(os.path.join(MODEL_DIR, "pytorch_model.bin"), map_location="cpu", weights_only=False)
raw = {re.sub(r"^(module\.|backbone\.|dinov2\.)+", "", k): v for k, v in raw.items()}
sd = {}
for i in range(len(bb.blocks)):
    p = f"encoder.layer.{i}."
    for part in ("weight", "bias"):
        sd[f"blocks.{i}.attn.qkv.{part}"] = torch.cat(
            [raw[p + f"attention.attention.{n}.{part}"] for n in ("query", "key", "value")], 0)
    for a, b in [("attention.output.dense.weight", "attn.proj.weight"),
                 ("attention.output.dense.bias", "attn.proj.bias"),
                 ("layer_scale1.lambda1", "ls1.gamma"), ("layer_scale2.lambda1", "ls2.gamma"),
                 ("norm1.weight", "norm1.weight"), ("norm1.bias", "norm1.bias"),
                 ("norm2.weight", "norm2.weight"), ("norm2.bias", "norm2.bias"),
                 ("mlp.fc1.weight", "mlp.fc1.weight"), ("mlp.fc1.bias", "mlp.fc1.bias"),
                 ("mlp.fc2.weight", "mlp.fc2.weight"), ("mlp.fc2.bias", "mlp.fc2.bias")]:
        sd[f"blocks.{i}.{b}"] = raw[p + a]
for a, b in [("embeddings.cls_token", "cls_token"), ("embeddings.position_embeddings", "pos_embed"),
             ("embeddings.patch_embeddings.projection.weight", "patch_embed.proj.weight"),
             ("embeddings.patch_embeddings.projection.bias", "patch_embed.proj.bias"),
             ("layernorm.weight", "norm.weight"), ("layernorm.bias", "norm.bias")]:
    sd[b] = raw[a]
gs = IMG_SIZE // bb.patch_embed.patch_size[0]
sd["pos_embed"] = resample_abs_pos_embed(sd["pos_embed"], [gs, gs], num_prefix_tokens=bb.num_prefix_tokens)
missing, _ = bb.load_state_dict(sd, strict=False)
assert not missing, f"DINOv2 weights did not load: {missing[:5]}"   # the one check worth keeping

class PoolWrap(nn.Module):
    """CLS + mean + top-k/8 mean over spatial positions. Handles both shapes timm returns with
    global_pool="": (B, tokens, C) for a ViT, (B, C, H, W) for a CNN, which has no CLS token."""
    def __init__(self, backbone, n_prefix):
        super().__init__()
        self.backbone, self.n_prefix = backbone, n_prefix
        self.cnn = not hasattr(backbone, "blocks")
    def forward(self, x):
        if self.cnn:
            x = x.contiguous(memory_format=torch.channels_last)
        t = self.backbone(x)
        if t.dim() == 4:
            p = t.flatten(2).transpose(1, 2)
            k = max(1, p.shape[1] // 8)
            return torch.cat([p.mean(1), p.topk(k, dim=1).values.mean(1)], -1)
        p = t[:, self.n_prefix:]
        k = max(1, p.shape[1] // 8)
        return torch.cat([t[:, 0], p.mean(1), p.topk(k, dim=1).values.mean(1)], -1)

bb = bb.to(DEV).eval()
for p in bb.parameters():
    p.requires_grad = False
NPRE, BLOCKS = bb.num_prefix_tokens, list(bb.blocks)
NB = len(BLOCKS)
MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEV).view(1, 3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225], device=DEV).view(1, 3, 1, 1)

def pixels_to_images(xb):
    """(B, slices, H, W) in [0,1] -> 2.5D RGB: each slice with its two neighbours as channels."""
    pv, nx = torch.cat([xb[:, :1], xb[:, :-1]], 1), torch.cat([xb[:, 1:], xb[:, -1:]], 1)
    return ((torch.stack([pv, xb, nx], 2).reshape(-1, 3, IMG_SIZE, IMG_SIZE)) - MEAN) / STD

with torch.no_grad():
    EMB_DIM = int(PoolWrap(bb, NPRE)(torch.randn(2, 3, IMG_SIZE, IMG_SIZE, device=DEV)).shape[-1])
print(f"{TIMM_NAME}: {NB} {'blocks' if TAG == 'dino' else 'stages'}, train top {UNFREEZE}, EMB_DIM {EMB_DIM}")


### Preprocessing
Slices ordered by `ImagePositionPatient` (filenames are SOP UIDs), fixed 130 mm crop,
laterality canonicalised per plane. Frozen features feed Phase A; pixels go to a JPEG cache.


In [ ]:
ORDER_TAGS = ["ImagePositionPatient", "ImageOrientationPatient", "InstanceNumber"]

class StudyPrep(Dataset):
    def __init__(self, ids, pick, lat, sz, ks):
        self.ids, self.pick, self.lat, self.sz, self.ks = ids, pick, lat, sz, ks
        self.ntk = sum(ks)
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        sid = self.ids[i]
        LT = self.lat.get(sid, "R")
        vol = np.zeros((self.ntk, self.sz, self.sz), np.uint8)
        mask = np.zeros(4, np.uint8)
        for sl in range(4):
            t0, K = int(np.sum(self.ks[:sl])), self.ks[sl]
            if (sid, sl) not in self.pick:
                continue
            info = self.pick[(sid, sl)]
            files = glob.glob(os.path.join(info["dir"], DCM_GLOB))
            if len(files) < 3:
                continue
            # Two-pass read. Filenames here are SOP UIDs and carry no spatial order, so
            # sorting by any number in them scrambles the volume. Pass 1 reads geometry
            # tags only and orders by ImagePositionPatient projected on the slice normal;
            # pass 2 decodes just the slices kept.
            # ~100 header reads per study, and each is network latency rather than CPU.
            # Latency-bound work wants more concurrency than cores, so thread inside the worker.
            with ThreadPoolExecutor(HDR_THREADS) as _ex:
                hdr = list(_ex.map(lambda f: pydicom.dcmread(
                    f, stop_before_pixels=True, force=True, specific_tags=ORDER_TAGS), files))
            pos = []
            for d in hdr:
                iop = getattr(d, "ImageOrientationPatient", None)
                ipp = getattr(d, "ImagePositionPatient", None)
                if iop is not None and ipp is not None and len(iop) == 6 and len(ipp) == 3:
                    nrm = np.cross(np.array(iop[:3], float), np.array(iop[3:], float))
                    pos.append(float(np.dot(np.array(ipp, float), nrm)))
                else:
                    pos.append(None)
            if all(p is not None for p in pos):
                order = sorted(range(len(files)),
                               key=lambda i: (pos[i], int(getattr(hdr[i], "InstanceNumber", 0) or 0)))
            else:
                order = sorted(range(len(files)),
                               key=lambda i: int(getattr(hdr[i], "InstanceNumber", i) or i))
            files = [files[i] for i in order]
            n = len(files)
            lo = int(0.10 * n); hi = max(int(0.90 * n) - 1, lo + 1)
            idxs = np.linspace(lo, hi, K).round().astype(int)
            if info["plane"] == "sagittal" and LT == "L":
                idxs = idxs[::-1]
            got = 0
            for kk, ii in enumerate(idxs):
                try:
                    d = pydicom.dcmread(files[int(ii)], force=True)
                    a = d.pixel_array.astype(np.float32)
                except Exception:
                    continue
                a = a * float(getattr(d, "RescaleSlope", 1) or 1) + \
                        float(getattr(d, "RescaleIntercept", 0) or 0)
                lv_, hv_ = np.percentile(a, 0.5), np.percentile(a, 99.5)
                a = np.clip((a - lv_) / max(hv_ - lv_, 1e-3), 0, 1)
                sp = getattr(d, "PixelSpacing", [1.0, 1.0])
                try:    spy, spx = float(sp[0]), float(sp[1])
                except Exception: spy = spx = 1.0
                hp = int(round(CROP_MM / max(spy, 1e-3))); wp = int(round(CROP_MM / max(spx, 1e-3)))
                Hh, Ww = a.shape; cy, cx = Hh // 2, Ww // 2
                c = a[max(0, cy - hp // 2):min(Hh, cy + hp // 2),
                      max(0, cx - wp // 2):min(Ww, cx + wp // 2)]
                if c.size == 0:
                    continue
                ph, pw = max(0, hp - c.shape[0]), max(0, wp - c.shape[1])
                if ph or pw:
                    c = np.pad(c, ((ph // 2, ph - ph // 2), (pw // 2, pw - pw // 2)))
                img = cv2.resize(c, (self.sz, self.sz), interpolation=cv2.INTER_AREA)
                if info["plane"] in ("axial", "coronal") and LT == "L":
                    img = np.ascontiguousarray(img[:, ::-1])
                vol[t0 + kk] = (img * 255).astype(np.uint8)
                got += 1
            if got >= K // 2:
                mask[sl] = 1
        return torch.from_numpy(vol), torch.from_numpy(mask)

pick, lat = {}, {}
for _, r in meta[meta.slot >= 0].iterrows():
    k = (r.sid, int(r.slot))
    if k not in pick or r.n > pick[k]["n"]:
        pick[k] = dict(dir=r.dir, n=int(r.n), plane=r.plane)
for sid, g in meta.groupby("sid"):
    lv = g.loc[g.L != "", "L"]
    lat[sid] = lv.mode().iloc[0] if len(lv) else "R"

ids = train_df[ID].tolist()
FEAT = np.zeros((len(ids), N_TOK, EMB_DIM), np.float16)
MASK = np.zeros((len(ids), 4), np.uint8)
JOFF = np.zeros(len(ids) * N_TOK + 1, np.int64)
cursor = done = 0
dp = nn.DataParallel(PoolWrap(bb, NPRE), device_ids=GPU_IDS)
with open(f"{WORK}/jpg_train.bin", "wb") as jf:
    for vols, masks in DataLoader(StudyPrep(ids, pick, lat, IMG_SIZE, K_PER_SLOT),
                                  batch_size=EXTRACT_BATCH, num_workers=NUM_WORKERS,
                                  pin_memory=True, prefetch_factor=4):
        B, vnp = vols.shape[0], vols.numpy()
        MASK[done:done + B] = masks.numpy()
        for b in range(B):
            for t in range(N_TOK):
                JOFF[(done + b) * N_TOK + t] = cursor
                if vnp[b, t].max() > 0:
                    enc = cv2.imencode(".jpg", vnp[b, t], [int(cv2.IMWRITE_JPEG_QUALITY), JPEG_Q])[1]
                    jf.write(enc.tobytes()); cursor += len(enc)
            JOFF[(done + b) * N_TOK + N_TOK] = cursor
        with torch.inference_mode(), torch.cuda.amp.autocast():
            f = dp(pixels_to_images(vols.to(DEV, non_blocking=True).float() / 255.))
        FEAT[done:done + B] = f.view(B, N_TOK, EMB_DIM).half().cpu().numpy()
        done += B
        if done % 800 < EXTRACT_BATCH:
            print(f"  {done}/{len(ids)}  {(time.time()-T0)/60:.0f} min")
JBLOB = np.memmap(f"{WORK}/jpg_train.bin", np.uint8, "r")
del dp
print(f"features {FEAT.shape} | jpeg {cursor/1e9:.1f} GB | slots {MASK.mean(0).round(2)}")


### Head
Per-label slot attention with an anatomical prior. Byte-identical to notebook C.


In [ ]:
SLOT_PRIOR_TABLE = {"ACL": (1, 3), "MCL": (2,), "Medial Meniscus": (1, 2, 3),
                    "Lateral Meniscus": (1, 2, 3), "Medial OA": (2, 3), "Lateral OA": (2, 3),
                    "PF OA": (0,), "Effusion": (0, 3), "Synovitis": (0, 3), "Baker's": (0, 3),
                    "Contusion": (0, 2, 3), "Fracture": (0, 1, 2, 3)}
SLOT_PRIOR = torch.zeros(NL, 4)
for _t, _sl in SLOT_PRIOR_TABLE.items():
    SLOT_PRIOR[LABELS.index(_t), list(_sl)] = PRIOR_W
SLOT_PRIOR = SLOT_PRIOR.to(DEV)

def head_logits(tk, mtok, H, tokmap, training=False):
    h = H["pr"](tk) + H["se"][tokmap][None]
    a = torch.einsum("bnh,oh->bon", h, H["q"]) / HEAD_HIDDEN ** 0.5
    a = a + SLOT_PRIOR[:, tokmap][None]
    a = torch.softmax(a.masked_fill(mtok.unsqueeze(1) < .5, -1e4), -1)
    ctx = torch.einsum("bon,bnh->boh", a, h)
    if MAX_MIX > 0:
        ctx = (1 - MAX_MIX) * ctx + MAX_MIX * torch.gather(
            h, 1, a.argmax(-1).unsqueeze(-1).expand(-1, -1, HEAD_HIDDEN))
    if training:
        ctx = F.dropout(ctx, HEAD_DROP, True)
    lg = (ctx * H["ow"][None]).sum(-1) + H["ob"]
    return lg + LAB_MIX * (torch.softmax(H["mx"], -1) @ lg.unsqueeze(-1)).squeeze(-1), h


### Training — (seed, fold) until the budget runs out
Each seed builds its own fold split. Per fold: Phase A trains the head on frozen
features, Phase B fine-tunes the top of the backbone from it. A fold that cannot finish is not
started; one cut by the budget is discarded, never saved.


In [ ]:
class JpegStudies(Dataset):
    def __init__(self, idx):
        self.idx = np.asarray(idx)
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        s = int(self.idx[i])
        vol = np.zeros((N_TOK, IMG_SIZE, IMG_SIZE), np.uint8)
        for t in range(N_TOK):
            o0, o1 = JOFF[s * N_TOK + t], JOFF[s * N_TOK + t + 1]
            if o1 > o0:
                vol[t] = cv2.imdecode(np.asarray(JBLOB[o0:o1]), cv2.IMREAD_GRAYSCALE)
        return torch.from_numpy(vol), s

def make_head():
    return {"pr": nn.Sequential(nn.LayerNorm(EMB_DIM), nn.Linear(EMB_DIM, HEAD_HIDDEN),
                                nn.GELU()).to(DEV),
            "se": nn.Parameter(torch.randn(4, HEAD_HIDDEN, device=DEV) * .02),
            "q":  nn.Parameter(torch.randn(NL, HEAD_HIDDEN, device=DEV) * .02),
            "ow": nn.Parameter(torch.randn(NL, HEAD_HIDDEN, device=DEV) * .02),
            "ob": nn.Parameter(torch.zeros(NL, device=DEV)),
            "mx": nn.Parameter(torch.eye(NL, device=DEV) * 3.)}

def head_params(H):
    return [p for v in H.values() for p in (v.parameters() if hasattr(v, "parameters") else [v])]

TK = torch.from_numpy(TOK2SLOT).to(DEV)
NS = len(train_df)
Yg, Wg = torch.from_numpy(soft).to(DEV), torch.from_numpy(sw).to(DEV)
Mg, Tg = torch.from_numpy(MASK.astype(np.float32)).to(DEV), torch.from_numpy(txt_t).to(DEV)
INIT_SD = {k: v.detach().clone() for k, v in bb.state_dict().items()}
# sort_index: the fold split must be identical in every session for resume to be valid
gsz, ggold = train_df.fp.value_counts().sort_index(), train_df.groupby("fp").is_gold.sum()
OOF, fold_est, stop = {}, FOLD_SECONDS, False
PREV = {os.path.basename(p): p for d in PREV_DIRS
        for p in glob.glob(f"{d}/**/m_{TAG}_s*_f*.pt", recursive=True)}

for SEED_I in SEEDS:
    # A different fold split per seed. Groups (scanner fingerprints) stay whole, so there is no
    # site leakage; the size-jittered ordering changes which groups share a fold, which makes
    # each seed's five members genuinely different rather than re-initialisations of the same five.
    rng = np.random.default_rng(SEED_I)
    order = sorted(gsz.index, key=lambda g: -gsz[g] * rng.uniform(0.75, 1.25))
    load, gload, assign = np.zeros(N_FOLDS), np.zeros(N_FOLDS), {}
    for g in order:
        f = int(np.lexsort((gload, load))[0])
        assign[g] = f; load[f] += gsz[g]; gload[f] += ggold.get(g, 0)
    fold_of = train_df.fp.map(assign).values.astype(int)
    oofA = np.full((NS, NL), np.nan, np.float32)
    oofB = np.full((NS, NL), np.nan, np.float32)
    for p in [p for d in PREV_DIRS for p in glob.glob(f"{d}/**/oof_{TAG}_s{SEED_I}.npy", recursive=True)]:
        o = np.load(p)                       # OOF of folds finished in an earlier session
        oofA, oofB = np.where(np.isnan(oofA), o[0], oofA), np.where(np.isnan(oofB), o[1], oofB)

    for FOLD in FOLDS:
        ckpt = f"{WORK}/m_{TAG}_s{SEED_I}_f{FOLD}.pt"
        if os.path.basename(ckpt) in PREV:     # done in an earlier session: carry it forward
            shutil.copy(PREV[os.path.basename(ckpt)], ckpt); continue
        # stop before a fold that cannot finish; fold_est tracks the slowest fold seen so far
        if time.time() - T0 + 1.1 * fold_est > TIME_BUDGET:
            print(f"  s{SEED_I} f{FOLD}: {(TIME_BUDGET-(time.time()-T0))/60:.0f} min left, "
                  f"a fold takes ~{fold_est/60:.0f} — stopping"); stop = True; break
        t_fold = time.time()
        torch.manual_seed(SEED_I * 100 + FOLD); np.random.seed(SEED_I * 100 + FOLD)
        tri, vai = np.where(fold_of != FOLD)[0], np.where(fold_of == FOLD)[0]

        # ---- Phase A: head on frozen features (~12 s) ----
        Fg = torch.from_numpy(FEAT).to(DEV).float()
        tr_t, va_t = torch.from_numpy(tri).to(DEV), torch.from_numpy(vai).to(DEV)
        H = make_head(); th = nn.Linear(HEAD_HIDDEN, 64).to(DEV)
        hp = head_params(H) + list(th.parameters())
        opt = torch.optim.AdamW(hp, lr=LR_HEAD, weight_decay=WD)
        sc = torch.optim.lr_scheduler.CosineAnnealingLR(opt, HEAD_EPOCHS)
        best, bstate = -1., None
        for ep in range(HEAD_EPOCHS):
            perm = tr_t[torch.randperm(len(tr_t), device=DEV)]
            for b0 in range(0, len(perm), 256):
                sel = perm[b0:b0 + 256]; mtok = Mg[sel][:, TK]
                lg, tk = head_logits(Fg[sel], mtok, H, TK, True)
                gpool = (tk * mtok.unsqueeze(-1)).sum(1) / mtok.sum(1, keepdim=True).clamp(min=1)
                loss = (F.binary_cross_entropy_with_logits(lg, Yg[sel], reduction="none") * Wg[sel]).mean() \
                     + TXT_W * (1. - (F.normalize(th(gpool), dim=-1) * Tg[sel]).sum(-1)).mean()
                opt.zero_grad(set_to_none=True); loss.backward()
                torch.nn.utils.clip_grad_norm_(hp, 3.); opt.step()
            sc.step()
            if ep % 5 == 4:
                with torch.no_grad():
                    vp = torch.sigmoid(head_logits(Fg[va_t], Mg[va_t][:, TK], H, TK)[0]).cpu().numpy()
                au = [roc_auc_score((soft[vai, j] > .5) * 1., vp[:, j]) for j in range(NL)
                      if 0 < (soft[vai, j] > .5).sum() < len(vai)]
                if np.mean(au) > best:
                    best, oofA[vai] = np.mean(au), vp
                    bstate = {k: (v.state_dict() if hasattr(v, "state_dict") else v.detach().clone())
                              for k, v in H.items()}
        del Fg, tr_t, va_t, H, th, opt; torch.cuda.empty_cache()

        # ---- Phase B: fine-tune the top of the backbone, head warm-started from Phase A ----
        bb.load_state_dict(INIT_SD)
        for p in bb.parameters():
            p.requires_grad = False
        for bi in range(NB - UNFREEZE, NB):
            for p in BLOCKS[bi].parameters():
                p.requires_grad = True
        for n, p in bb.named_parameters():
            if n.startswith(("norm", "head.norm")):
                p.requires_grad = True
        bp = [p for p in bb.parameters() if p.requires_grad]

        H = make_head()
        H["pr"].load_state_dict(bstate["pr"])
        for k in ("se", "q", "ow", "ob", "mx"):
            H[k] = nn.Parameter(bstate[k].clone().to(DEV))
        hp = head_params(H)
        grp = [{"params": list(BLOCKS[bi].parameters()), "lr": LR_BB * (0.75 ** (NB - 1 - bi))}
               for bi in range(NB - UNFREEZE, NB)]
        grp += [{"params": [p for n, p in bb.named_parameters()
                            if n.startswith(("norm", "head.norm")) and p.requires_grad], "lr": LR_BB},
                {"params": hp, "lr": LR_HEAD * .1}]
        grp = [g for g in grp if len(g["params"])]
        opt = torch.optim.AdamW(grp, weight_decay=WD)
        tl = DataLoader(JpegStudies(tri), batch_size=BATCH_STUDIES, shuffle=True, drop_last=True,
                        num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True,
                        prefetch_factor=4)
        vl = DataLoader(JpegStudies(vai), batch_size=BATCH_STUDIES * 2, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
        sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[g["lr"] for g in grp],
                                                  total_steps=max(1, PHASE_B_EPOCHS * len(tl) // ACCUM),
                                                  pct_start=.15, div_factor=10., final_div_factor=100.)
        scaler = torch.cuda.amp.GradScaler()
        ema = ModelEmaV3(bb, decay=EMA_DECAY, use_warmup=True, device=DEV)
        dp = nn.DataParallel(PoolWrap(bb, NPRE), device_ids=GPU_IDS)
        cut = False

        for ep in range(PHASE_B_EPOCHS):
            bb.train(); opt.zero_grad(set_to_none=True)
            for it, (xb_u8, sel) in enumerate(tl):
                sel = sel.to(DEV)
                xb = xb_u8.to(DEV, non_blocking=True).float() / 255.
                B = xb.shape[0]; mb = Mg[sel].clone()
                xb = xb.clamp(1e-4, 1).pow(torch.empty(B, 1, 1, 1, device=DEV).uniform_(.8, 1.25))
                xb = xb + torch.randn_like(xb) * .01
                sh = int(IMG_SIZE * .04)
                xb = torch.roll(xb, (np.random.randint(-sh, sh + 1), np.random.randint(-sh, sh + 1)), (2, 3))
                mb = mb * (torch.rand(B, 4, device=DEV) > .10).float()
                mb[:, 0] = torch.clamp(mb[:, 0] + (mb.sum(1) == 0).float(), max=1.)
                with torch.cuda.amp.autocast():
                    tk0 = dp(pixels_to_images(xb)).view(B, N_TOK, EMB_DIM)
                    lg, _ = head_logits(tk0, mb[:, TK], H, TK)
                    loss = (F.binary_cross_entropy_with_logits(lg, Yg[sel], reduction="none")
                            * Wg[sel]).mean()
                scaler.scale(loss / ACCUM).backward()
                if (it + 1) % ACCUM == 0:
                    scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(bp + hp, 3.)
                    scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
                    ema.update(bb)
                    if sch.last_epoch < sch.total_steps - 1:
                        sch.step()
                if it % 200 == 0:
                    print(f"  s{SEED_I} f{FOLD} e{ep} {it}/{len(tl)} loss {loss.item():.4f} "
                          f"(floor {FLOOR:.3f}) {torch.cuda.max_memory_allocated()/1e9:.1f}GB "
                          f"{(time.time()-T0)/60:.0f}m")
                if time.time() - T0 > TIME_BUDGET:
                    cut = True; break
            if cut:
                break

        if cut:
            # a half-trained fold would enter the ensemble looking like a finished one — discard it
            print(f"  s{SEED_I} f{FOLD}: cut by the time budget, NOT saved"); stop = True
            del dp, ema, opt; torch.cuda.empty_cache(); break

        ema.module.eval()
        dpe = nn.DataParallel(PoolWrap(ema.module, NPRE), device_ids=GPU_IDS)
        vp = np.zeros((len(vai), NL), np.float32); off = 0
        with torch.inference_mode():
            for xb_u8, sel in vl:
                xb = xb_u8.to(DEV).float() / 255.
                B = xb.shape[0]
                with torch.cuda.amp.autocast():
                    tk0 = dpe(pixels_to_images(xb)).view(B, N_TOK, EMB_DIM)
                    lg, _ = head_logits(tk0, Mg[sel.to(DEV)][:, TK], H, TK)
                vp[off:off + B] = torch.sigmoid(lg.float()).cpu().numpy(); off += B
        oofB[vai] = vp
        torch.save({"timm_name": TIMM_NAME, "emb": EMB_DIM, "img": IMG_SIZE, "tag": TAG,
                    "seed": SEED_I, "fold": FOLD,
                    "bb": {k: v.half() for k, v in ema.module.state_dict().items()},
                    **{k: (v.state_dict() if hasattr(v, "state_dict") else v.detach().cpu())
                       for k, v in H.items()}}, ckpt)
        fold_est = max(fold_est, time.time() - t_fold)
        print(f"  s{SEED_I} f{FOLD} saved | {(time.time()-t_fold)/60:.0f} min/fold "
              f"| {(time.time()-T0)/60:.0f} min total")
        del dp, dpe, ema, opt, tl, vl; torch.cuda.empty_cache()

    OOF[SEED_I] = (oofA, oofB)
    if not np.isnan(oofB).all():
        np.save(f"{WORK}/oof_{TAG}_s{SEED_I}.npy", np.stack([oofA, oofB]))
    if stop:
        break

del JBLOB
os.remove(f"{WORK}/jpg_train.bin")     # 5.4 GB pixel cache; C re-reads DICOM for the test set
print(f"{len(glob.glob(f'{WORK}/m_{TAG}_s*_f*.pt'))} checkpoints | {(time.time()-T0)/60:.0f} min")


### Out-of-fold evaluation
`LB ≈ 0.899 × gold + 0.113`, fitted on five submissions.


In [ ]:
# Gold is what predicts the leaderboard: LB ~= 0.899 x gold + 0.113 across five submissions.
# Soft is only the in-fold selection metric. Each seed has its own OOF over the same 58 gold
# studies, so averaging their ranks gives the seed-ensemble OOF — the gain from more members.
def gold_auc(pred):
    gm = (~np.isnan(pred[:, 0])) & (train_df.is_gold.values == 1)
    a = [roc_auc_score(train_df.loc[gm, L].values.astype(float), pred[gm, j])
         for j, L in enumerate(LABELS) if 0 < train_df.loc[gm, L].sum() < gm.sum()]
    return (np.mean(a) if a else float("nan")), int(gm.sum())

full = []
for s, (oa, ob) in OOF.items():
    if np.isnan(ob).all():
        continue
    ga, na = gold_auc(oa); gb, nb_ = gold_auc(ob)
    print(f"seed {s:3d}  Phase A gold {ga:.4f} (n={na})   Phase B gold {gb:.4f} (n={nb_})"
          f"   -> LB ~{0.899 * gb + 0.113:.3f}")
    if not np.isnan(ob[:, 0]).any():
        full.append(ob)

if len(full) > 1:
    ens = np.mean([np.apply_along_axis(lambda c: pd.Series(c).rank(pct=True).values, 0, o)
                   for o in full], 0)
    g, n = gold_auc(ens)
    print(f"\n{len(full)}-seed rank ensemble  gold {g:.4f} (n={n})   -> LB ~{0.899 * g + 0.113:.3f}")
